In [1]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout[:300])

# Dùng subprocess.run thay !pip magic → tương thích cả notebook lẫn .py script
subprocess.run(['pip', 'uninstall', 'torchaudio', 'torchcodec', '-y', '-q'], check=False)
subprocess.run(['pip', 'install', 'torch==2.3.1',
                '--index-url', 'https://download.pytorch.org/whl/cu121', '-q'], check=True)
subprocess.run(['pip', 'install',
                'datasets==2.20.0', 'transformers==4.41.0',
                'librosa==0.10.2', 'scipy==1.13.1', 'soundfile==0.12.1',  # FIX v11: pin versions tránh breaking changes
                '-q'], check=True)

import os, random, warnings, time
warnings.filterwarnings('ignore')
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import soundfile as sf
import librosa
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from datasets import load_dataset
from scipy.stats import pearsonr

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'  # AMP chỉ hữu ích trên GPU
print(f'torch: {torch.__version__}')
print(f'GPU:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'AMP:   {"enabled" if USE_AMP else "disabled (CPU mode)"}')
try:
    x = torch.randn(4,4).cuda(); _ = x @ x.T; print('CUDA: OK ✓')
except Exception as e:
    print(f'CUDA FAIL: {e}')
print(f'Device: {DEVICE}')


Sat May  2 14:23:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 50.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/12

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.5.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.


torch: 2.3.1+cu121
GPU:   Tesla T4
AMP:   enabled
CUDA: OK ✓
Device: cuda


In [2]:
# ── 1. Load dataset ───────────────────────────────────────────────────────────
from datasets import load_dataset
print('Loading speechocean762...')
train_ds = load_dataset('mispeech/speechocean762', split='train')
test_ds  = load_dataset('mispeech/speechocean762', split='test')
print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

s = train_ds[0]
print('Fields:', list(s.keys()))
print(f'accuracy={s["accuracy"]} fluency={s["fluency"]} prosodic={s["prosodic"]} total={s["total"]}')

for field in ['accuracy','fluency','prosodic','total']:
    vals = [float(d[field]) for d in train_ds]
    print(f'{field}: min={min(vals):.1f} max={max(vals):.1f} mean={np.mean(vals):.2f} std={np.std(vals):.2f}')

# Assert label scale đúng 0-10 — phát hiện sớm nếu dataset thay đổi format
for field in ['accuracy', 'fluency', 'prosodic', 'total']:
    max_val = max(float(d[field]) for d in train_ds)
    assert max_val <= 10.0, (
        f"Label '{field}' có max={max_val:.1f} > 10 — kiểm tra lại scale dataset!"
        " Nếu dataset dùng thang 0-5, sửa labels / 5.0 thay vì / 10.0 trong SpeechOceanDataset."
    )
print('✓ Label scale OK: tất cả fields nằm trong [0, 10]')


Loading speechocean762...


Generating train split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Train: 2500 | Test: 2500
Fields: ['accuracy', 'completeness', 'fluency', 'prosodic', 'text', 'total', 'words', 'speaker', 'gender', 'age', 'audio']
accuracy=8 fluency=9 prosodic=9 total=8
accuracy: min=3.0 max=10.0 mean=7.62 std=1.69
fluency: min=1.0 max=10.0 mean=7.73 std=1.55
prosodic: min=1.0 max=10.0 mean=7.42 std=1.50
total: min=2.0 max=10.0 mean=7.18 std=1.67
✓ Label scale OK: tất cả fields nằm trong [0, 10]


In [3]:
# ── 2. Dataset & Collate ──────────────────────────────────────────────────────
MODEL_NAME     = 'facebook/wav2vec2-base-960h'
PROCESSOR_NAME = 'facebook/wav2vec2-base-960h'
MAX_LEN_SEC    = 15
BATCH          = 8

processor = Wav2Vec2Processor.from_pretrained(PROCESSOR_NAME)

class SpeechOceanDataset(Dataset):
    def __init__(self, ds, augment=False):
        self.ds      = ds
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item  = self.ds[idx]
        audio = np.array(item['audio']['array'], dtype=np.float32)
        sr    = item['audio']['sampling_rate']

        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

        audio = audio[:16000 * MAX_LEN_SEC]

        if self.augment:
            if random.random() < 0.4:
                audio = audio + np.random.randn(len(audio)).astype(np.float32) * 0.003
            if random.random() < 0.5:
                audio = audio * random.uniform(0.8, 1.2)
            if random.random() < 0.3:
                rate  = random.uniform(0.95, 1.05)
                # FIX v11: truncate trước khi stretch tránh allocate array dài rồi cắt
                # rate<1.0 → audio dài hơn sau stretch → cắt thừa ~5% samples
                # pre-truncate giảm memory + CPU đáng kể khi MAX_LEN_SEC=15
                pre_len = int(16000 * MAX_LEN_SEC * rate) + 1
                audio = librosa.effects.time_stretch(audio[:pre_len], rate=rate)
                audio = audio[:16000 * MAX_LEN_SEC]

        peak  = np.abs(audio).max()
        audio = audio / peak if peak > 1e-6 else np.zeros_like(audio)

        labels = np.array([
            float(item['accuracy']),
            float(item['fluency']),
            float(item['prosodic']),
            float(item['total']),
        ], dtype=np.float32)

        if self.augment:
            # noise độc lập cho từng aspect (acc/flu/pro), KHÔNG corrupt total
            # FIX: Gaussian σ=0.2 thay uniform ±0.5
            # uniform ±0.5 (~5% thang 0-10) làm aspect scores lệch quá xa total
            # Gaussian σ=0.2: 95% noise nằm trong ±0.4 — nhỏ hơn, thực tế hơn
            noise = np.random.normal(0, 0.2, size=3).astype(np.float32)
            labels[:3] = np.clip(labels[:3] + noise, 0, 10)

        return audio, torch.tensor(labels / 10.0, dtype=torch.float32)


def collate_fn(batch):
    audios, labels = zip(*batch)
    enc = processor(
        list(audios), sampling_rate=16000,
        return_tensors='pt', padding=True, return_attention_mask=True,
    )
    return enc.input_values, enc.attention_mask, torch.stack(labels)


# num_workers=0: tránh CUDA multiprocess conflict trên Kaggle GPU (chữ đỏ)
# pin_memory bỏ: không cần khi num_workers=0
train_loader = DataLoader(SpeechOceanDataset(train_ds, augment=True),
                          batch_size=BATCH, shuffle=True, collate_fn=collate_fn,
                          num_workers=0)
test_loader  = DataLoader(SpeechOceanDataset(test_ds, augment=False),
                          batch_size=BATCH, shuffle=False, collate_fn=collate_fn,
                          num_workers=0)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')
av, am, lb = next(iter(train_loader))
print(f'Audio: {av.shape} | Labels: {lb[0]}')
print(f'Audio NaN: {torch.isnan(av).any()} | Label NaN: {torch.isnan(lb).any()}')


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Train batches: 313 | Test batches: 313


2026-05-02 14:26:50.143274: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777732010.339307      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777732010.392940      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777732010.863028      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777732010.863061      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777732010.863064      22 computation_placer.cc:177] computation placer alr

Audio: torch.Size([8, 108944]) | Labels: tensor([0.8099, 0.9972, 0.9130, 0.8000])
Audio NaN: False | Label NaN: False


In [4]:
# ── 3. Model — wav2vec2-base, unfreeze 6/12 layers ─────────────────────────────

def _wav2vec2_feat_lengths(input_lengths: torch.Tensor, w2v_config) -> torch.Tensor:
    """
    Tính output length của CNN feature extractor từ input sample length.
    đọc conv_kernel / conv_stride từ model.config thay vì hardcode
    → an toàn khi đổi sang wav2vec2-large hoặc model khác.
    """
    kernel_sizes = w2v_config.conv_kernel
    strides      = w2v_config.conv_stride
    lengths = input_lengths
    for k, s in zip(kernel_sizes, strides):
        lengths = (lengths - k) // s + 1
    return lengths.clamp(min=0)


class PronunciationScorer(nn.Module):
    def __init__(self, unfreeze_last_n=6, n_layers_avg=4):
        super().__init__()
        self.w2v          = Wav2Vec2Model.from_pretrained(MODEL_NAME)
        self.n_layers_avg = n_layers_avg

        # Freeze CNN hoàn toàn
        for p in self.w2v.feature_extractor.parameters():
            p.requires_grad = False

        # Freeze toàn bộ transformer trước
        for p in self.w2v.encoder.parameters():
            p.requires_grad = False

        # Unfreeze 6 layers cuối (base có 12 layers → unfreeze nửa sau)
        n_layers = len(self.w2v.encoder.layers)
        print(f'Total transformer layers: {n_layers} | Unfreezing last {unfreeze_last_n}')
        for i in range(n_layers - unfreeze_last_n, n_layers):
            for p in self.w2v.encoder.layers[i].parameters():
                p.requires_grad = True

        # Unfreeze feature projection
        for p in self.w2v.feature_projection.parameters():
            p.requires_grad = True

        # learnable weights để blend n_layers_avg hidden states cuối
        self.layer_weights = nn.Parameter(torch.ones(n_layers_avg) / n_layers_avg)

        H = 768  # wav2vec2-base hidden size

        # Attention pooling thay mean pooling
        # Model học tự trọng số frame nào quan trọng cho pronunciation
        # → tập trung vào frame phonetically rich, bỏ qua silence/padding
        self.attn_pool = nn.Linear(H, 1)

        self.trunk = nn.Sequential(
            nn.LayerNorm(H),
            nn.Linear(H, 256),
            nn.GELU(),
            nn.Dropout(0.15),
        )

        def head():
            return nn.Sequential(
                nn.Linear(256, 64), nn.GELU(),
                nn.Linear(64, 1),   nn.Sigmoid()
            )
        self.h_acc = head(); self.h_flu = head()
        self.h_pro = head(); self.h_tot = head()

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    def forward(self, input_values, attention_mask=None):
        out = self.w2v(
            input_values,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # Weighted average n layers cuối
        stacked = torch.stack(out.hidden_states[-self.n_layers_avg:], dim=0)  # [K, B, T, 768]
        weights = torch.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
        hidden  = (stacked * weights).sum(0)   # [B, T, 768]

        # Attention pooling với mask
        # Thay vì mean pooling, model học trọng số từng frame
        # → tập trung vào frame phonetically rich, bỏ qua silence/padding
        if attention_mask is not None:
            feat_lengths = _wav2vec2_feat_lengths(attention_mask.sum(-1), self.w2v.config).long()
            T    = hidden.shape[1]
            # FIX: vectorized pad_mask thay Python for-loop trên batch
            # torch.arange so sánh với feat_lengths → tạo mask [B, T] không cần loop
            # clamp(max=T) guard length==0 và length>T cùng lúc
            idx      = torch.arange(T, device=hidden.device).unsqueeze(0)        # [1, T]
            pad_mask = (idx < feat_lengths.clamp(max=T).unsqueeze(1)).float()    # [B, T]
            # attn_logits: [B, T, 1] → mask padding thành -inf trước softmax
            attn_logits  = self.attn_pool(hidden)                                # [B, T, 1]
            attn_logits  = attn_logits + (1.0 - pad_mask.unsqueeze(-1)) * -1e9
            attn_weights = torch.softmax(attn_logits, dim=1)                     # [B, T, 1]
            pooled = (hidden * attn_weights).sum(1)                              # [B, H]
        else:
            attn_logits  = self.attn_pool(hidden)                             # [B, T, 1]
            attn_weights = torch.softmax(attn_logits, dim=1)
            pooled = (hidden * attn_weights).sum(1)                           # [B, H]

        z = self.trunk(pooled)
        return torch.stack([
            self.h_acc(z).squeeze(-1),
            self.h_flu(z).squeeze(-1),
            self.h_pro(z).squeeze(-1),
            self.h_tot(z).squeeze(-1),
        ], dim=-1)


model = PronunciationScorer(unfreeze_last_n=6, n_layers_avg=4).to(DEVICE)

# Sanity check
model.eval()
with torch.no_grad():
    out = model(av.to(DEVICE), am.to(DEVICE))
    print(f'Output shape: {out.shape}')
    print(f'Output NaN:   {torch.isnan(out).any()}')
    print(f'Output range: {out.min():.4f} ~ {out.max():.4f}')


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total transformer layers: 12 | Unfreezing last 6
Params: 43,188,233 / 94,636,937 (45.6%)
Output shape: torch.Size([8, 4])
Output NaN:   False
Output range: 0.4466 ~ 0.5137


In [5]:
# ── 4. Loss: 0.6 × Huber + 0.4 × PCC ────────────────────────────────────────
# Huber: robust với outlier (người nói quá tốt/tệ)
# PCC loss: optimize trực tiếp Pearson Correlation — metric chính khi eval
# Kết hợp 2 loss → cả magnitude lẫn ranking đều tốt

huber = nn.HuberLoss(delta=0.5)

def pcc_loss(pred, target):
    """1 - mean PCC qua 4 aspects. Minimize → maximize correlation."""
    # FIX v11: guard batch quá nhỏ (batch cuối epoch có thể 1-2 sample)
    # variance ≈ 0 → corr = 0/1e-6 → loss spike → gradient bất thường
    if pred.shape[0] < 4:
        return torch.zeros(1, device=pred.device, requires_grad=True).squeeze()
    losses = []
    for i in range(pred.shape[1]):
        p = pred[:, i];  t = target[:, i]
        p_c = p - p.mean();  t_c = t - t.mean()
        # 1e-8 quá nhỏ với float32 — khi batch có constant labels (t_c=0)
        # gradient explode. Dùng 1e-6 ổn định hơn, sai số PCC không đáng kể.
        corr = (p_c * t_c).sum() / (
            torch.sqrt((p_c**2).sum() * (t_c**2).sum()) + 1e-6
        )
        # clamp corr về (-1+eps, 1-eps) — tránh gradient =0 khi model collapse
        # (predict constant → p_c≈0 → corr≈0/eps → gradient rất nhỏ → training đình trệ)
        corr = corr.clamp(-1.0 + 1e-6, 1.0 - 1e-6)
        losses.append(1.0 - corr)
    return torch.stack(losses).mean()

def criterion(pred, target):
    return 0.6 * huber(pred, target) + 0.4 * pcc_loss(pred, target)

model.train()
with torch.no_grad():
    preds = model(av.to(DEVICE), am.to(DEVICE))
    loss  = criterion(preds, lb.to(DEVICE))
    print(f'Initial loss: {loss.item():.4f}  (Huber + PCC, không được NaN)')
    print(f'Preds NaN: {torch.isnan(preds).any()}')


Initial loss: 0.4514  (Huber + PCC, không được NaN)
Preds NaN: False


In [6]:
# ── 5. Train — AMP + warmup + per-epoch checkpoint + auto-resume ─────────────
NUM_EPOCHS    = 30
WARMUP_EPOCHS = 3          # linear warmup bảo vệ pretrained weights
LR_BACKBONE   = 5e-6
LR_HEAD       = 8e-5
GRAD_CLIP     = 0.5
PATIENCE      = 8
ACCUM_STEPS   = 4          # effective batch = 8×4 = 32

SAVE_PATH = '/kaggle/working/pron_scorer_best.pt'
CKPT_PATH = '/kaggle/working/pron_scorer_latest.pt'  # resume nếu session die
ASPECT    = ['Accuracy', 'Fluency', 'Prosodic', 'Total']

optimizer = AdamW([
    {'params': list(model.w2v.parameters()),          'lr': LR_BACKBONE, 'weight_decay': 0.01},
    {'params': [model.layer_weights],                 'lr': LR_HEAD},
    # FIX: attn_pool bị thiếu → weights không bao giờ được update → attention pooling không học
    {'params': list(model.attn_pool.parameters()),    'lr': LR_HEAD},
    {'params': list(model.trunk.parameters()),        'lr': LR_HEAD,     'weight_decay': 0.01},
    {'params': list(model.h_acc.parameters()),        'lr': LR_HEAD},
    {'params': list(model.h_flu.parameters()),        'lr': LR_HEAD},
    {'params': list(model.h_pro.parameters()),        'lr': LR_HEAD},
    {'params': list(model.h_tot.parameters()),        'lr': LR_HEAD},
], eps=1e-8)

# SequentialLR thay LambdaLR + CosineAnnealingLR thủ công.
# BUG cũ (trước đây): CosineAnnealingLR khởi tạo ngay từ đầu, âm thầm đếm qua 3 epoch warmup
# → khi cosine_scheduler.step() được gọi lần đầu ở epoch 4 nó đã 'tiêu' 3 epoch của T_max
# → LR decay bị lệch pha hoàn toàn.
# SequentialLR chuyển đúng sang cosine SAU khi warmup kết thúc (milestone=WARMUP_EPOCHS).
#
# LinearLR start_factor=1/WARMUP_EPOCHS:
# epoch 1: LR × 0.333 | epoch 2: LR × 0.667 | epoch 3: LR × 1.0 (warmup thật sự từ đầu)
# trước đây dùng LambdaLR epoch/WARMUP_EPOCHS → epoch 0 = LR×0, mất 1 epoch gradient.
# LinearLR, SequentialLR đã được import ở cell 0

warmup_scheduler = LinearLR(
    optimizer,
    start_factor = 1.0 / WARMUP_EPOCHS,
    end_factor   = 1.0,
    total_iters  = WARMUP_EPOCHS,
)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max   = NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min = 1e-8,
)
scheduler = SequentialLR(
    optimizer,
    schedulers = [warmup_scheduler, cosine_scheduler],
    milestones = [WARMUP_EPOCHS],
)

# GradScaler với keyword arg device= (tương thích PyTorch >= 2.4)
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=USE_AMP)  # dùng DEVICE.type thay hardcode 'cuda'

def evaluate():
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        # đổi tên biến b_av/b_am/b_lb tránh shadow outer scope av/am/lb
        for b_av, b_am, b_lb in test_loader:
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                p = model(b_av.to(DEVICE), b_am.to(DEVICE))
            preds_all.append(p.cpu().numpy())
            labels_all.append(b_lb.numpy())
    P = np.concatenate(preds_all)  * 10
    L = np.concatenate(labels_all) * 10
    pccs, mses = {}, {}
    for i, name in enumerate(ASPECT):
        pcc, _     = pearsonr(P[:,i], L[:,i])
        pccs[name] = pcc
        mses[name] = np.mean((P[:,i]-L[:,i])**2)
    return pccs, mses

# ── Auto-resume nếu session bị kill ──────────────────────────────────────────
best_pcc   = -1.0
best_epoch = 0
no_improve = 0
START      = 1

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)  # False vì cần optimizer/scheduler state
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])  # 1 key thay 2
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    START      = ckpt['epoch'] + 1
    best_pcc   = ckpt['best_pcc']
    best_epoch = ckpt['best_epoch']
    no_improve = ckpt['no_improve']
    print(f"✓ Resumed từ epoch {ckpt['epoch']} | best PCC: {best_pcc:.4f}")
else:
    print('Starting fresh training...')

# ── Training loop ─────────────────────────────────────────────────────────────
print(f'Training {NUM_EPOCHS} epochs ({WARMUP_EPOCHS} warmup) | model: {MODEL_NAME}')
print(f"{'Ep':>3} | {'Loss':>7} | {'acc':>6} {'flu':>6} {'pro':>6} {'tot':>6} | {'avg':>6} | {'no_imp':>6} | {'t':>4}")
print('-'*78)

for epoch in range(START, NUM_EPOCHS + 1):
    model.train()
    losses = []
    t0 = time.time()
    optimizer.zero_grad()

    # t_av/t_am/t_lb: tránh shadow biến av/am/lb ở outer scope (cell 2 sanity check)
    for step, (t_av, t_am, t_lb) in enumerate(train_loader):
        t_av = t_av.to(DEVICE); t_am = t_am.to(DEVICE); t_lb = t_lb.to(DEVICE)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            pred = model(t_av, t_am)
            loss = criterion(pred, t_lb) / ACCUM_STEPS

        # check NaN TRƯỚC backward — backward trên NaN loss làm corrupt gradient
        # FIX v11: bỏ zero_grad() khi NaN — zero_grad xóa cả gradient hợp lệ
        # đã tích lũy từ các micro-step trước trong cùng accumulation window
        # Chỉ skip micro-step NaN này, giữ nguyên gradient đã tích lũy
        if torch.isnan(loss):
            print(f"⚠️  NaN loss tại step {step}, epoch {epoch} — skip micro-step")
            continue

        scaler.scale(loss).backward()
        losses.append(loss.item() * ACCUM_STEPS)

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    scheduler.step()  # 1 lệnh, SequentialLR tự biết phase warmup/cosine
    avg_loss = float(np.mean(losses)) if losses else float('nan')
    pccs, _  = evaluate()
    avg_pcc  = float(np.mean(list(pccs.values())))
    elapsed  = time.time() - t0

    flag = ''
    if avg_pcc > best_pcc:
        best_pcc   = avg_pcc
        best_epoch = epoch
        no_improve = 0
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':    scaler.state_dict(),
            'best_pcc':             best_pcc,
            'best_epoch':           best_epoch,
            # FIX v11: lưu đầy đủ state vào SAVE_PATH
            # trước đây thiếu optimizer/scheduler → không fine-tune được từ best
            # nếu CKPT_PATH bị xóa (Kaggle session reset), best model vẫn resume được
        }, SAVE_PATH)
        flag = ' ★'
    else:
        no_improve += 1

    # Checkpoint mỗi epoch → không mất progress nếu session die
    torch.save({
        'epoch': epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),  # 1 key thay 2 key cũ
        'scaler_state_dict':    scaler.state_dict(),
        'best_pcc':   best_pcc,
        'best_epoch': best_epoch,
        'no_improve': no_improve,
    }, CKPT_PATH)

    phase      = 'warmup' if epoch <= WARMUP_EPOCHS else 'cosine'
    current_lr = optimizer.param_groups[0]['lr']  # FIX v11: log LR để debug scheduler
    print(f"{epoch:>3} | {avg_loss:>7.4f} | "
          f"{pccs['Accuracy']:>6.4f} {pccs['Fluency']:>6.4f} "
          f"{pccs['Prosodic']:>6.4f} {pccs['Total']:>6.4f} | "
          f"{avg_pcc:>6.4f}{flag} | {no_improve:>6} | {elapsed/60:.1f}m [{phase}] lr={current_lr:.2e}")

    if no_improve >= PATIENCE:
        print(f'Early stop @ epoch {epoch} (no improve {PATIENCE} epochs)')
        break

print(f'\nBest Avg PCC: {best_pcc:.4f} @ epoch {best_epoch}')

print(f'Best model : {SAVE_PATH}')


Starting fresh training...
Training 30 epochs (3 warmup) | model: facebook/wav2vec2-base-960h
 Ep |    Loss |    acc    flu    pro    tot |    avg | no_imp |    t
------------------------------------------------------------------------------
  1 |  0.3311 | 0.4580 0.3434 0.4269 0.4392 | 0.4169 ★ |      0 | 2.0m [warmup] lr=2.78e-06
  2 |  0.1901 | 0.5729 0.6210 0.6246 0.5879 | 0.6016 ★ |      0 | 2.1m [warmup] lr=3.89e-06
  3 |  0.1423 | 0.5838 0.6361 0.6381 0.6028 | 0.6152 ★ |      0 | 2.1m [warmup] lr=5.00e-06
  4 |  0.1325 | 0.5884 0.6541 0.6580 0.6076 | 0.6270 ★ |      0 | 2.0m [cosine] lr=4.98e-06
  5 |  0.1184 | 0.6063 0.6749 0.6758 0.6296 | 0.6466 ★ |      0 | 2.1m [cosine] lr=4.93e-06
  6 |  0.1161 | 0.6053 0.6718 0.6734 0.6274 | 0.6445 |      1 | 2.1m [cosine] lr=4.85e-06
  7 |  0.1015 | 0.6109 0.6830 0.6846 0.6351 | 0.6534 ★ |      0 | 2.1m [cosine] lr=4.73e-06
  8 |  0.1038 | 0.6112 0.6807 0.6814 0.6335 | 0.6517 |      1 | 2.0m [cosine] lr=4.59e-06
  9 |  0.1024 | 0.6200 0.6

In [7]:
# ── 6. Final eval ─────────────────────────────────────────────────────────────
# FIX v9: guard trước khi load — cell 6 hay bị chạy độc lập khi session resume
# mà SAVE_PATH chưa tồn tại (model chưa đủ epoch để lưu best, hoặc cell 5 chưa chạy)
import os as _os

_best_exists   = _os.path.exists(SAVE_PATH)
_latest_exists = _os.path.exists(CKPT_PATH)

if not _best_exists and not _latest_exists:
    raise FileNotFoundError(
        f'Không tìm thấy checkpoint nào.\n'
        f'  best   : {SAVE_PATH}\n'
        f'  latest : {CKPT_PATH}\n'
        f'→ Hãy chạy Cell 5 (training) trước, hoặc upload file .pt vào /kaggle/working/'
    )

# Ưu tiên best model; fallback sang latest nếu chưa có best (early stop chưa trigger)
_load_path = SAVE_PATH if _best_exists else CKPT_PATH
if not _best_exists:
    print(f'⚠ best model chưa có, load từ latest checkpoint: {_load_path}')

# FIX: weights_only=False vì SAVE_PATH chứa cả epoch, best_pcc (Python primitives)
# weights_only=True trong PyTorch>=2.0 raise UnpicklingError với non-tensor values
ckpt = torch.load(_load_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✓ Loaded model từ: {_load_path}')
if 'best_pcc' in ckpt:
    print(f'  best PCC @ epoch {ckpt.get("epoch", "?")}: {ckpt["best_pcc"]:.4f}')

# ── Collect all predictions ───────────────────────────────────────────────────
model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    # FIX v11: đổi tên e_av/e_am/e_lb tránh shadow biến av/am/lb của cell 2/3
    # nếu dùng tên av/am/lb, chạy lại cell 3 sau cell 6 sẽ dùng batch test thay batch train
    for e_av, e_am, e_lb in test_loader:
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p = model(e_av.to(DEVICE), e_am.to(DEVICE))
        preds_all.append(p.cpu().numpy())
        labels_all.append(e_lb.numpy())

P = np.concatenate(preds_all)  * 10   # [N, 4] scale 0-10
L = np.concatenate(labels_all) * 10

# ── 1. PCC ────────────────────────────────────────────────────────────────────
pccs = {}
for i, name in enumerate(ASPECT):
    pcc, _ = pearsonr(P[:,i], L[:,i])
    pccs[name] = pcc

# ── 2. MAE / RMSE ─────────────────────────────────────────────────────────────
maes  = {name: np.mean(np.abs(P[:,i] - L[:,i]))      for i, name in enumerate(ASPECT)}
rmses = {name: np.mean((P[:,i] - L[:,i])**2)**0.5    for i, name in enumerate(ASPECT)}

# ── 3. IELTS Band accuracy ────────────────────────────────────────────────────
def to_band(scores):
    """
    Quy về IELTS band 0-9, làm tròn 0.5.

    Mapping: speechocean762 score [0-10] → IELTS band [0-9]
    Công thức: band = score / 10 * 9  (tỉ lệ tuyến tính)
    Căn cứ: IELTS Pronunciation sub-score 0-9 tương ứng với
    speechocean762 total score 0-10 theo linear assumption.
    Đây là approximation — không có mapping chính thức từ dataset.
    """
    b = np.clip(scores, 0, 10) / 10 * 9
    return np.round(b * 2) / 2

P_band = to_band(P)
L_band = to_band(L)

exact_match = {}
within_half = {}
within_one  = {}
for i, name in enumerate(ASPECT):
    diff = np.abs(P_band[:,i] - L_band[:,i])
    exact_match[name] = np.mean(diff == 0.0)
    within_half[name] = np.mean(diff <= 0.5)
    within_one[name]  = np.mean(diff <= 1.0)

# ── Print report ──────────────────────────────────────────────────────────────
print()
print('=' * 70)
print('FINAL EVALUATION — wav2vec2-base | 6L unfrozen | attention pooling + weighted hidden states')
print('=' * 70)

print(f'\n{"── 1. Correlation & Error":}')
print(f'  {"Aspect":<12} {"PCC":>7} {"MAE":>7} {"RMSE":>7}  {"Status"}')
print(f'  {"-"*50}')
for name in ASPECT:
    pcc  = pccs[name]
    mae  = maes[name]
    rmse = rmses[name]
    status = '✓ good' if pcc >= 0.70 else ('~ ok' if pcc >= 0.60 else '✗ weak')
    print(f'  {name:<12} {pcc:>7.4f} {mae:>7.4f} {rmse:>7.4f}  {status}')

avg_pcc  = np.mean(list(pccs.values()))
avg_mae  = np.mean(list(maes.values()))
avg_rmse = np.mean(list(rmses.values()))
print(f'  {"─"*50}')
print(f'  {"Average":<12} {avg_pcc:>7.4f} {avg_mae:>7.4f} {avg_rmse:>7.4f}')
print(f'  {"GOPT SOTA":<12} {"0.7430":>7}   {"~0.6":>7}   {"~0.8":>7}')

print(f'\n{"── 2. IELTS Band Accuracy (thang 0-9, làm tròn 0.5):"}')
print(f'  {"Aspect":<12} {"Exact":>8} {"±0.5 band":>10} {"±1.0 band":>10}')
print(f'  {"-"*45}')
for name in ASPECT:
    print(f'  {name:<12} {exact_match[name]:>7.1%} {within_half[name]:>10.1%} {within_one[name]:>10.1%}')

avg_exact = np.mean(list(exact_match.values()))
avg_half  = np.mean(list(within_half.values()))
avg_one   = np.mean(list(within_one.values()))
print(f'  {"─"*45}')
print(f'  {"Average":<12} {avg_exact:>7.1%} {avg_half:>10.1%} {avg_one:>10.1%}')
print(f'\n  → Cathoven claim: ~98% within 0.5 band (dùng nhiều data hơn)')
print(f'  → Mục tiêu tốt:    ≥70% within 0.5 band')

print('=' * 70)

# ── 4. Sample predictions ─────────────────────────────────────────────────────
print(f'\n── Sample predictions (10 mẫu đầu test set):')
print(f'  {"#":>3} {"P_acc":>6} {"T_acc":>6} | {"P_flu":>6} {"T_flu":>6} | {"P_tot":>6} {"T_tot":>6} | {"Band_P":>7} {"Band_T":>7}')
print(f'  {"-"*65}')
for i in range(10):
    print(f'  {i+1:>3} {P[i,0]:>6.1f} {L[i,0]:>6.1f} | '
          f'{P[i,1]:>6.1f} {L[i,1]:>6.1f} | '
          f'{P[i,3]:>6.1f} {L[i,3]:>6.1f} | '
          f'{P_band[i,3]:>7.1f} {L_band[i,3]:>7.1f}')


✓ Loaded model từ: /kaggle/working/pron_scorer_best.pt
  best PCC @ epoch 23: 0.6931

FINAL EVALUATION — wav2vec2-base | 6L unfrozen | attention pooling + weighted hidden states

── 1. Correlation & Error
  Aspect           PCC     MAE    RMSE  Status
  --------------------------------------------------
  Accuracy      0.6451  0.9086  1.2521  ~ ok
  Fluency       0.7305  0.7455  1.0347  ✓ good
  Prosodic      0.7268  0.7323  1.0171  ✓ good
  Total         0.6700  0.8643  1.2157  ~ ok
  ──────────────────────────────────────────────────
  Average       0.6931  0.8126  1.1299
  GOPT SOTA     0.7430      ~0.6      ~0.8

── 2. IELTS Band Accuracy (thang 0-9, làm tròn 0.5):
  Aspect          Exact  ±0.5 band  ±1.0 band
  ---------------------------------------------
  Accuracy       22.2%      53.3%      81.5%
  Fluency        27.1%      63.5%      87.8%
  Prosodic       26.7%      65.6%      88.5%
  Total          22.6%      61.7%      82.6%
  ─────────────────────────────────────────────


In [8]:
# ── 7. WhisperX Word-level — CHẠY RIÊNG, không chạy cùng cell train ─────────
#
# WhisperX conflict với torch==2.3.1 trên Kaggle.
# Cách dùng đúng:
# 1. Download pron_scorer_best.pt về máy local
# 2. Tạo môi trường riêng:
# conda create -n whisperx python=3.10
# pip install torch==2.1.0 whisperx
# 3. Chạy hàm get_word_level_scores() dưới đây trên local
#
# Hoặc dùng Kaggle notebook KHÁC (kernel riêng không cài torch==2.3.1)

WHISPERX_CODE = '''
import whisperx, torch, numpy as np

# ── Singleton cache: tránh load lại model mỗi lần gọi ─────────────────────
# load 1 lần duy nhất → giảm latency từ ~8s xuống ~50ms per call
_WX_CACHE = {}

def _get_whisperx_models(device: str):
    if 'whisper' not in _WX_CACHE:
        compute_type = 'float16' if device == 'cuda' else 'int8'
        _WX_CACHE['whisper']  = whisperx.load_model('large-v2', device, compute_type=compute_type)
        align_model, metadata = whisperx.load_align_model(language_code='en', device=device)
        _WX_CACHE['align']    = align_model
        _WX_CACHE['metadata'] = metadata
    return _WX_CACHE['whisper'], _WX_CACHE['align'], _WX_CACHE['metadata']


def _score_word_segment(audio_full, start_s, end_s, pron_model, proc, device):
    """
    Chạy PronunciationScorer trên đoạn audio [start_s, end_s].
    Trả về accuracy score [0, 1] (đã normalize).
    kết hợp acoustic model vào word-level scoring
    thay vì chỉ dùng WhisperX alignment confidence.
    """
    start_sample = int(start_s * 16000)
    end_sample   = int(end_s   * 16000)
    word_audio   = audio_full[start_sample:end_sample]

    if len(word_audio) < 1600:  # < 0.1s → quá ngắn
        return -1.0

    peak = np.abs(word_audio).max()
    if peak > 1e-6:
        word_audio = word_audio / peak

    try:
        enc = proc(word_audio, sampling_rate=16000,
                   return_tensors='pt', return_attention_mask=True)
        # FIX: eval() để tắt Dropout → scores deterministic
        # FIX: autocast để nhất quán với cách model được train (AMP)
        pron_model.eval()
        use_amp = (device == 'cuda')
        with torch.no_grad():
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                scores = pron_model(
                    enc.input_values.to(device),
                    enc.attention_mask.to(device),
                )
        return float(scores[0, 0].cpu())  # accuracy index 0, range [0,1]
    except Exception:
        return -1.0


def get_word_level_scores(audio_path, pron_model=None, proc=None):
    """
      - Singleton cache: không load lại model mỗi lần gọi
      - Acoustic blending: blend alignment conf + PronunciationScorer per word
      - Dynamic thresholds: dựa trên percentile thực tế thay vì hardcode
      - return_char_alignments=True: giữ char data cho phoneme hint sau này
      - highlighted_html có wrapper <p> để frontend nhúng dễ hơn
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    whisper_model, align_model, metadata = _get_whisperx_models(device)
    audio  = whisperx.load_audio(audio_path)
    result = whisper_model.transcribe(audio, batch_size=16)

    # return_char_alignments=True để có char-level data
    result = whisperx.align(
        result['segments'], align_model, metadata, audio, device,
        return_char_alignments=True
    )

    word_scores = []
    for seg in result['segments']:
        for w in seg.get('words', []):
            align_conf = round(float(w.get('score', 1.0)), 3)
            start_s    = float(w.get('start', 0))
            end_s      = float(w.get('end',   0))

            # blend alignment + acoustic
            if pron_model is not None and proc is not None:
                # FIX v11: pron_model.to(device) thay next(...).to(device)
                # .to() trên tensor trả về tensor mới, không in-place → model vẫn ở CPU
                # pron_model.to(device) mới thực sự move toàn bộ parameters
                pron_model.to(device)
                acoustic = _score_word_segment(audio, start_s, end_s, pron_model, proc, device)
            else:
                acoustic = -1.0

            if acoustic >= 0.0:
                blended_conf = round(0.4 * align_conf + 0.6 * acoustic, 3)
            else:
                blended_conf = align_conf

            word_scores.append({
                'word':           w.get('word', '').strip(),
                'start':          round(start_s, 3),
                'end':            round(end_s,   3),
                'confidence':     blended_conf,
                'align_conf':     align_conf,
                'acoustic_score': round(acoustic, 3) if acoustic >= 0 else None,
            })

    # Dynamic thresholds dựa trên phân phối thực tế
    # Tránh edge case: tất cả từ 'bad' khi user phát âm tệ, hoặc tất cả 'ok' khi tốt
    confs = np.array([w['confidence'] for w in word_scores])
    if len(confs) >= 5:
        THRESHOLD_BAD  = float(np.clip(np.percentile(confs, 20), 0.30, 0.60))
        THRESHOLD_WARN = float(np.clip(np.percentile(confs, 50), 0.55, 0.80))
    else:
        THRESHOLD_BAD, THRESHOLD_WARN = 0.50, 0.70

    annotated_tokens = []
    for w in word_scores:
        conf  = w['confidence']
        level = 'bad' if conf < THRESHOLD_BAD else ('warn' if conf < THRESHOLD_WARN else 'ok')
        annotated_tokens.append({
            'word':           w['word'],
            'start':          w['start'],
            'end':            w['end'],
            'confidence':     conf,
            'align_conf':     w['align_conf'],
            'acoustic_score': w['acoustic_score'],
            'level':          level,
        })

    _STYLE = {
        'ok':   '',
        'warn': 'color:#BA7517;background:#FAEEDA;border-radius:3px;padding:1px 4px;font-weight:600',
        'bad':  'color:#A32D2D;background:#FCEBEB;border-radius:3px;padding:1px 4px;font-weight:600;text-decoration:underline wavy #E24B4A',
    }
    html_parts = []
    for tok in annotated_tokens:
        style = _STYLE[tok['level']]
        if style:
            ac  = tok['acoustic_score']
            tip = f"conf={tok['confidence']:.2f}" + (f" | acoustic={ac:.2f}" if ac is not None else "")
            html_parts.append(f'<span style="{style}" title="{tip}">{tok["word"]}</span>')
        else:
            html_parts.append(tok['word'])

    # wrapper <p> để frontend nhúng dễ hơn
    highlighted_html = (
        '<p class="transcript-highlight" '
        'style="line-height:2;font-size:16px;font-family:sans-serif">'
        + ' '.join(html_parts)
        + '</p>'
    )

    transcript = ' '.join(w['word'] for w in word_scores)
    weak_words = [t for t in annotated_tokens if t['level'] != 'ok']

    return {
        'transcript':        transcript,
        'word_scores':       word_scores,
        'annotated_tokens':  annotated_tokens,
        'highlighted_html':  highlighted_html,
        'weak_words':        weak_words,
        'thresholds':        {'bad': round(THRESHOLD_BAD, 3), 'warn': round(THRESHOLD_WARN, 3)},
    }
'''

# WHISPERX_CODE được exec() vì whisperx chỉ khả dụng trong môi trường riêng.
# compile() thay exec() thuần: traceback trỏ đúng dòng trong WHISPERX_CODE thay vì "<string>".
# Để dùng local: lưu WHISPERX_CODE ra file whisperx_scorer.py rồi import trực tiếp.
#   with open("whisperx_scorer.py", "w") as f: f.write(WHISPERX_CODE)
try:
    import whisperx  # noqa
    exec(compile(WHISPERX_CODE, "<whisperx_cell>", "exec"), globals())
    print('✓ WhisperX có sẵn — get_word_level_scores() đã được define.')
except ImportError:
    # Bình thường trên Kaggle — xem hướng dẫn môi trường ở đầu cell
    def get_word_level_scores(*args, **kwargs):  # stub: cell 8 dùng ImportError để bắt
        raise ImportError("WhisperX chưa được cài. Xem comment ở đầu cell 7.")
    print('WhisperX không khả dụng trong môi trường này.')
    print('Xem hướng dẫn môi trường trong comment ở đầu cell.')


WhisperX không khả dụng trong môi trường này.
Xem hướng dẫn môi trường trong comment ở đầu cell.


In [9]:
# ── 8. Full pipeline — sentence score (luôn chạy) + word-level (nếu có) ──────

def full_ielts_score(
    audio_path: str,
    _model=None,
    _processor=None,
) -> dict:
    """
    Input:  đường dẫn file .wav từ user
    Output: dict đầy đủ để hiển thị trên frontend

    sentence_scores luôn có.
    word_analysis chỉ có nếu WhisperX đã được cài (môi trường local).
    latency_ms: thời gian inference (ms) để benchmark deployment.

    nhận _model/_processor làm tham số thay vì phụ thuộc global
    → có thể import/test sang module khác mà không lỗi NameError.
    Nếu không truyền vào, fallback về global model/processor (tương thích ngược).
    truyền model + processor vào get_word_level_scores()
    → word-level dùng acoustic score thay vì chỉ alignment confidence.
    bỏ import soundfile bên trong function → dùng sf đã import ở cell 0.
    """
    # fallback về global nếu không truyền tham số (tương thích ngược)
    _model     = _model     if _model     is not None else model
    _processor = _processor if _processor is not None else processor
    audio, sr = sf.read(audio_path)
    if audio.ndim == 2: audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if sr != 16000: audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    audio = audio[:16000 * MAX_LEN_SEC]
    if len(audio) == 0:
        raise ValueError(f'Audio rỗng sau khi truncate: {audio_path}')
    peak = np.abs(audio).max()
    if peak > 1e-6: audio = audio / peak

    _model.eval()
    enc = _processor(audio, sampling_rate=16000, return_tensors='pt',
                     padding=True, return_attention_mask=True)

    t_start = time.time()
    with torch.no_grad():
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            scores = _model(enc.input_values.to(DEVICE),
                            enc.attention_mask.to(DEVICE)).cpu().float().numpy()[0] * 10
    latency_ms = (time.time() - t_start) * 1000

    sentence_scores = {
        'accuracy':   round(float(scores[0]), 1),
        'fluency':    round(float(scores[1]), 1),
        'prosodic':   round(float(scores[2]), 1),
        'total':      round(float(scores[3]), 1),
        'ielts_band': float(np.clip(np.round((scores[3] / 10 * 9) * 2) / 2, 0, 9)),
    }

    # WhisperX optional — chỉ chạy nếu cài được
    # truyền model + processor để blend acoustic score per word
    word_result = None
    try:
        import whisperx  # noqa
        word_result = get_word_level_scores(
            audio_path,
            pron_model=_model,      # : dùng tham số thay global
            proc=_processor,
        )
    except ImportError:
        pass  # bình thường trên Kaggle; stub ở cell 7 cũng raise ImportError
    # NameError không còn xảy ra: cell 7 luôn define stub khi whisperx vắng mặt

    return {
        'sentence_scores': sentence_scores,
        'word_analysis':   word_result,
        'latency_ms':      round(latency_ms, 1),
    }


def print_word_analysis(word_analysis: dict):
    """In transcript với từng từ được đánh màu rõ ràng theo level."""
    tokens = word_analysis['annotated_tokens']
    thresh = word_analysis.get('thresholds', {'bad': 0.50, 'warn': 0.70})
    print(f'\n=== TRANSCRIPT (từng từ) — ngưỡng: bad<{thresh["bad"]:.2f} warn<{thresh["warn"]:.2f} ===')
    line_parts = []
    for tok in tokens:
        w = tok['word']
        if tok['level'] == 'bad':
            line_parts.append(f'[\u2717{w.upper()}]')
        elif tok['level'] == 'warn':
            line_parts.append(f'(~{w})')
        else:
            line_parts.append(w)
    print(' '.join(line_parts))

    weak = word_analysis['weak_words']
    if weak:
        print(f'\n\u2500\u2500 Từ cần chú ý ({len(weak)} từ):')
        for tok in weak:
            mark = '\u2717 SAI       ' if tok['level'] == 'bad' else '~ CẢI THIỆN'
            ac   = tok.get('acoustic_score')
            ac_str = f' | acoustic={ac:.2f}' if ac is not None else ''
            print(f'  {mark}  "{tok["word"]}"  (conf={tok["confidence"]:.2f}{ac_str})'
                  f'  [{tok["start"]:.2f}s \u2013 {tok["end"]:.2f}s]')
    else:
        print('\n\u2713 Tất cả từ phát âm đạt ngưỡng.')


# ── Test với 1 mẫu từ test set ────────────────────────────────────────────
import tempfile  # soundfile đã import là sf ở cell 0

sample = test_ds[0]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
    sf.write(tmp.name, audio_array, 16000)
    tmp_path = tmp.name

result = full_ielts_score(tmp_path, _model=model, _processor=processor)

print('=== SENTENCE SCORES ===')
for k, v in result['sentence_scores'].items():
    print(f'  {k:<12}: {v}')
print(f'  {"latency_ms":<12}: {result["latency_ms"]} ms')

print()
print('=== GROUND TRUTH ===')
print(f'  accuracy  : {sample["accuracy"]}')
print(f'  fluency   : {sample["fluency"]}')
print(f'  prosodic  : {sample["prosodic"]}')
print(f'  total     : {sample["total"]}')

if result['word_analysis']:
    print_word_analysis(result['word_analysis'])
    print('\n\u2500\u2500 highlighted_html (nhúng thẳng vào frontend):')
    print(result['word_analysis']['highlighted_html'][:400], '...')
    print(f'\n\u2500\u2500 Thresholds động: {result["word_analysis"]["thresholds"]}')
else:
    print()
    print('(WhisperX không khả dụng — chạy trên local để có word-level analysis)')

import os; os.unlink(tmp_path)


=== SENTENCE SCORES ===
  accuracy    : 8.9
  fluency     : 8.6
  prosodic    : 8.5
  total       : 8.4
  ielts_band  : 7.5
  latency_ms  : 146.7 ms

=== GROUND TRUTH ===
  accuracy  : 9
  fluency   : 9
  prosodic  : 9
  total     : 9

(WhisperX không khả dụng — chạy trên local để có word-level analysis)


In [10]:
# ── 9. Export ─────────────────────────────────────────────────────────────────
import os
print('Files để download từ Kaggle:')
if not os.path.exists(SAVE_PATH):
    print(f'  ⚠ {SAVE_PATH} chưa tồn tại — hãy chạy Cell 5 (training) trước.')
else:
    size_mb = os.path.getsize(SAVE_PATH) / 1e6
    print(f'  {SAVE_PATH}')
    print(f'  Size: {size_mb:.0f} MB')
print()
print('Deploy trên server:')
print('  pip install torch transformers librosa soundfile whisperx')
print('  python score_audio.py --audio user.wav')


Files để download từ Kaggle:
  /kaggle/working/pron_scorer_best.pt
  Size: 724 MB

Deploy trên server:
  pip install torch transformers librosa soundfile whisperx
  python score_audio.py --audio user.wav
